# Helmholtz near-field jump relations

This notebook studies the one-sided near-field evaluation of the Helmholtz single- and double-layer potentials and compares numerical integration strategies through their jump and continuity properties.

The key idea is to examine **both layer potentials separately**, without solving an underlying boundary value problem. The fixed FE surface and FE densities define the experiment itself, so **geometry approximation and density interpolation errors do not enter these tests**. This makes it possible to isolate the near-field integration behavior of each potential.


### Description of the experiment

The finite element surface $\Gamma_h$ is the geometry of the experiment, and $m_h$ and $j_h$ are fixed interpolated FE densities. We select a point $\boldsymbol y_0$ strictly inside one surface element, where the outward element normal $\boldsymbol n_h$ is unambiguous, and approach it symmetrically from both sides:

$$
\boldsymbol x_\delta^{\rm out}=\boldsymbol y_0+\delta\boldsymbol n_h,
\qquad
\boldsymbol x_\delta^{\rm in}=\boldsymbol y_0-\delta\boldsymbol n_h,
\qquad \delta\to0^+.
$$

The jump and continuity relations hold for these discrete densities on $\Gamma_h$ itself. We therefore do not measure how accurately $\Gamma_h$, $m_h$, and $j_h$ approximate an analytical geometry and analytical densities; those errors require a separate refinement study. At each finite distance, a jump defect contains both the quadrature error and the physical approach-to-the-limit contribution.

#### Double-layer potential: jump

For the double-layer potential $D$ and the sign convention used here,

$$
D(m_h)(\boldsymbol x_\delta^{\rm out})
-D(m_h)(\boldsymbol x_\delta^{\rm in})
\longrightarrow m_h(\boldsymbol y_0).
$$

The relative jump defect is

$$
E_D(\delta)=
\frac{\left|D(m_h)(\boldsymbol x_\delta^{\rm out})-D(m_h)(\boldsymbol x_\delta^{\rm in})-m_h(\boldsymbol y_0)\right|}
{|m_h(\boldsymbol y_0)|}.
$$

#### Single-layer potential: jump in the normal derivative

The normal derivative of the single-layer potential has a jump. Using the **same outward normal** $\boldsymbol n_h$ on both sides, define

$$
q_\delta^{\rm out/in}
=\boldsymbol n_h\cdot\nabla S(j_h)(\boldsymbol x_\delta^{\rm out/in}).
$$

Then

$$
q_\delta^{\rm out}-q_\delta^{\rm in}\longrightarrow-j_h(\boldsymbol y_0),
\qquad
E_{\partial_n S}(\delta)=
\frac{\left|q_\delta^{\rm out}-q_\delta^{\rm in}+j_h(\boldsymbol y_0)\right|}
{|j_h(\boldsymbol y_0)|}.
$$

The normal remains fixed: the jump is in the normal component of the potential gradient. In plain text, the relative defect is `abs(q_out - q_in + j_h(y0)) / abs(j_h(y0))`.

#### Single-layer potential: continuity and its limitations

The single-layer potential itself is continuous across $\Gamma_h$:

$$
S(j_h)(\boldsymbol x_\delta^{\rm out})
-S(j_h)(\boldsymbol x_\delta^{\rm in})
\longrightarrow0.
$$

Its normalized continuity defect is

$$
E_S(\delta)=
\frac{\left|S(j_h)(\boldsymbol x_\delta^{\rm out})-S(j_h)(\boldsymbol x_\delta^{\rm in})\right|}
{\max\{|S(j_h)(\boldsymbol x_\delta^{\rm out})|,|S(j_h)(\boldsymbol x_\delta^{\rm in})|,10^{-15}\}}.
$$

**Continuity can be tested, but a small continuity defect alone does not establish quadrature accuracy.** The symmetric approach can produce nearly equal errors on both sides, which cancel in the difference even when the individual potential values are inaccurate. Moreover, a fixed Gauss sum is itself continuous away from its quadrature nodes and can therefore satisfy this test while missing the correct limiting values.

The double-layer jump and the jump in the single-layer normal derivative provide the more informative jump criteria. Accuracy of the single-layer values must be assessed separately, by comparing each side with a numerical reference at the same finite distance; this avoids cancellation between the two sides.


#### Numerical quadratures

With the FE geometry and densities fixed, we compare three numerical integration strategies, distinguished by superscripts:

- $*$: **singularity extraction** (also labelled **analytical + Duffy**).
- $G$: **naive Gauss** — ordinary Gaussian quadrature on every surface element, applied to the full integrand.
- $D$: **Duffy only** — Duffy quadrature for the full integrand on nearby elements, and ordinary Gaussian quadrature on distant elements. Nearness is determined relative to the local element size $h$; nearby elements are split into reference triangles around the projected target point before applying the Duffy transformation.

All three methods use the same absolute quadrature order for each layer, although their quadrature point counts differ. The orders are `2 + flux_order + bonus` for SL and its normal derivative, and `2 + trace_order + bonus` for DL. At fixed order, Duffy only can still underresolve the sharply peaked DL and SL normal-derivative kernels at very small approach distances.


#### Helmholtz data and diagnostics

The analytical Helmholtz trace and the negative of its outward normal derivative are interpolated into complex FE spaces to obtain $m_h$ and $j_h$, respectively. The criteria above are tested for these fixed discrete data.

The main jump plots use the SL normal-derivative jump. The continuity defect is retained in `single_layer_continuity_defect` as a secondary check. Here `q_side = dot(normal_h, grad(sl)(point_side))`. The built-in evaluator differentiates the kernel analytically; the notebook does not approximate `grad(sl)` by finite differences of SL values.

#### Numerical reference and single-layer value errors

The exact jump criteria do not require a numerical reference. For the separate SL value analysis, we compute one singularity-extraction reference at each target pair using the single `reference_bonus_intorder` fixed at the beginning of the notebook. This measures quadrature differences on the same discrete problem, not certified errors against exact potential values.

The additional **SL value-error plots** compare the exterior and interior
values individually against this reference on the same curved mesh, with the
same FE density and target coordinates. For each side, the plotted error is

```text
abs(SL_side - reference_SL_side) / max(abs(reference_SL_side), 1e-15)
```

This comparison avoids cancellation between the two sides. The
`sl_value_reference_error` column records the larger of the two relative value
errors, while the plots show the exterior and interior errors separately.

#### Duffy implementation and output

The shared [duffy_reference.py](duffy_reference.py) helper constructs Duffy rules
in Python and passes the full SL/DL and SL normal-derivative integrands to NGSolve's `Integrate`.
It uses the same near-element criterion and five-step constrained Gauss-Newton
projection as `bem/potentialcf.cpp`: nearby triangles/quads are split into up to
three/four reference triangles at the projected point; distant elements keep
ordinary Gauss quadrature. The original curved geometry and FE densities are
used throughout. No target mesh or C++ option is needed.

Results are computed on each run, kept in the notebook's DataFrames, and
exported to CSV in `output/`. The Duffy comparison is computed in Python
and does not require precomputed CSV input.



In [ ]:
from pathlib import Path
from duffy_reference import DuffyQuadrature

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import Markdown, display
import numpy as np
import pandas as pd
from netgen.occ import OCCGeometry, Sphere
from ngsolve import BND, TRIG, CF, GridFunction, H1, Integrate, Mesh, SurfaceL2, TaskManager, grad, ds, exp, specialcf, sqrt, x, y, z
from ngsolve.fem import IntegrationRule
from ngsolve.bem import HelmholtzDL, HelmholtzSL

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOUR_PI = 4.0 * np.pi
radius = 1.0
maxh = 0.2
curve_order = 3  # configurable; fixed during one experiment run
trace_order = 3
flux_order = trace_order - 1
source_point = np.array((0.20, -0.15, 0.10), dtype=float)
desired_direction = np.array((1.0, 1.0, 1.0), dtype=float)
desired_direction /= np.linalg.norm(desired_direction)
kappa = 1.0  # Helmholtz wave number
distances = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
bonus_intorders = list(range(5, 22, 2))
selected_bonus_intorder = bonus_intorders[-1]
reference_bonus_intorder = 80
bem_params = {'use_fmm': False}

## Experiment profile



In [ ]:
display(Markdown(rf"""

| Quantity | Configuration in this experiment |
|---|---|
| FE geometry $\Gamma_h$ | fixed to `maxh = {maxh}` |
| Geometry order | fixed to `curve_order = {curve_order}`; configurable before a run |
| Pole $\boldsymbol x_s$ | fixed at `{tuple(float(value) for value in source_point)}` |
| Element search direction | fixed at `{tuple(float(value) for value in desired_direction)}`; configurable before a run |
| Trace spaces | fixed to `H1(order={trace_order})`, `SurfaceL2(order={flux_order})` |
| Cauchy data | fixed FE densities `m_h`$\in$ `H1(order={trace_order})` and `j_h`$\in$ `SurfaceL2(order={flux_order})` |
| Approach sides | inside and outside, evaluated symmetrically |
| Quadrature method | varied: naive Gauss ($G$), Duffy only ($D$), analytical + Duffy ($*$) |
| Quadrature bonuses | varied: `{bonus_intorders}` |
| Effective SL and grad(SL) / DL orders | `2 + flux_order + bonus` / `2 + trace_order + bonus`, for all methods |
| Approach distances | varied: `{distances}` |
| Bonus shown in distance plot | `selected_bonus_intorder = {selected_bonus_intorder}` |
| SL value reference | singularity extraction, `bonus_intorder={reference_bonus_intorder}` |
"""))


In [ ]:
sphere = Sphere((0, 0, 0), radius)
sphere.faces.name = 'sphere'
source_mesh = Mesh(OCCGeometry(sphere).GenerateMesh(maxh=maxh)).Curve(curve_order)
source_region = source_mesh.Boundaries('sphere')
duffy_quadrature = DuffyQuadrature(source_mesh, source_region)

boundary_elements = [element for element in source_mesh.Elements(BND) if len(element.vertices) == 3]
centroid_rule = IntegrationRule(TRIG, 1)  # one point at (1/3, 1/3)
mapped_centroids = source_mesh.MapToAllElements(centroid_rule, BND)
mapped_coordinates = np.column_stack((
    np.asarray(x(mapped_centroids)).reshape(-1),
    np.asarray(y(mapped_centroids)).reshape(-1),
    np.asarray(z(mapped_centroids)).reshape(-1),
))
if len(boundary_elements) != len(mapped_coordinates):
    raise ValueError('the current experiment expects a purely triangular surface mesh')

selected_index = max(
    range(len(boundary_elements)),
    key=lambda index: np.dot(
        mapped_coordinates[index] / np.linalg.norm(mapped_coordinates[index]),
        desired_direction,
    ),
)
selected_element = boundary_elements[selected_index]
selected_vertices = np.array(
    [source_mesh[vertex].point for vertex in selected_element.vertices],
    dtype=float,
)
surface_mesh_point = mapped_centroids[selected_index]
surface_point = np.array(
    (x(surface_mesh_point), y(surface_mesh_point), z(surface_mesh_point)),
    dtype=float,
)
normal_h = np.array(specialcf.normal(3)(surface_mesh_point), dtype=float)
normal_h /= np.linalg.norm(normal_h)
if np.dot(normal_h, surface_point) < 0:
    normal_h *= -1.0

boundary_triangles = [
    (element, np.array([source_mesh[vertex].point for vertex in element.vertices], dtype=float), mapped_coordinates[index])
    for index, element in enumerate(boundary_elements)
]
print(f'selected boundary element: {selected_element.nr}')
print(f'element-interior point: {tuple(surface_point)}')
print(f'outward element normal: {tuple(normal_h)}')


In [ ]:
distance_to_source = sqrt(
    (x-source_point[0])**2 + (y-source_point[1])**2 + (z-source_point[2])**2
)
helmholtz_trace = exp(1j * kappa * distance_to_source) / distance_to_source
helmholtz_gradient = CF((
    helmholtz_trace.Diff(x),
    helmholtz_trace.Diff(y),
    helmholtz_trace.Diff(z),
))
mesh_normal = specialcf.normal(3)

trace_space = H1(
    source_mesh,
    order=trace_order,
    complex=True,
    definedon=source_region,
)
flux_space = SurfaceL2(
    source_mesh,
    order=flux_order,
    complex=True,
    dual_mapping=False,
    definedon=source_region,
)
m_h = GridFunction(trace_space, name='m_h')
j_h = GridFunction(flux_space, name='j_h')
with TaskManager():
    m_h.Set(helmholtz_trace, definedon=source_region)
    j_h.Set(-(helmholtz_gradient * mesh_normal), definedon=source_region)

# exact trace at surface point, used for relative derivative-jump test
m_at_surface = complex(m_h(surface_mesh_point))
if abs(m_at_surface) < 1e-14:
    raise ValueError('m_h is too small at the selected point for a relative jump test')
print(f'm_h(y0) = {m_at_surface:.12e}')

# exact flux at surface point, used for relative derivative-jump test
j_at_surface = complex(j_h(surface_mesh_point))
if abs(j_at_surface) < 1e-14:
    raise ValueError('j_h is too small at the selected point for a relative derivative-jump test')
print(f'j_h(y0) = {j_at_surface:.12e}')


## Symmetric approach points

The selected triangle is highlighted in red. The inner and outer points shown below use a visible representative distance; the dashed segment follows the normal of the mapped FE element at $\boldsymbol y_0$. Here $\boldsymbol y_0$ is the image of the barycentric point of the reference triangle under the element map, and therefore lies strictly inside the element. The background triangulation is a schematic rendering through the element vertices. The figure is saved as `output/helmholtz_jump_approach_points.png`.

In [ ]:
visualization_delta = 5e-2
point_out_vis = surface_point + visualization_delta * normal_h
point_in_vis = surface_point - visualization_delta * normal_h
all_triangles = [vertices for _, vertices, _ in boundary_triangles]

fig = plt.figure(figsize=(7.2, 6.2))
ax = fig.add_subplot(111, projection='3d')
ax.add_collection3d(Poly3DCollection(all_triangles, facecolor='lightsteelblue', edgecolor='gray', linewidth=0.25, alpha=0.22))
ax.add_collection3d(Poly3DCollection([selected_vertices], facecolor='crimson', edgecolor='darkred', linewidth=1.2, alpha=0.8))
ax.plot(*np.column_stack((point_in_vis, point_out_vis)), color='black', linestyle='--', linewidth=1.3, label='element normal')
ax.scatter(*surface_point, color='black', s=35, label=r'$y_0$')
ax.scatter(*point_out_vis, color='tab:green', s=55, label=r'$x_\delta^{out}$')
ax.scatter(*point_in_vis, color='tab:orange', s=55, label=r'$x_\delta^{in}$')
ax.set_xlim(surface_point[0]-0.25, surface_point[0]+0.25)
ax.set_ylim(surface_point[1]-0.25, surface_point[1]+0.25)
ax.set_zlim(surface_point[2]-0.25, surface_point[2]+0.25)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_zlabel('$z$')
ax.set_title(
    f'Symmetric approach'
)
ax.legend(loc='best')
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "helmholtz_jump_approach_points.png", dpi=300)
plt.show()


In [ ]:
def evaluate_star_pair(point_out, point_in, bonus_intorder):
    """Evaluate SL, DL, and the SL normal derivative at physical coordinates."""
    m_trial = trace_space.TrialFunction()
    j_trial = flux_space.TrialFunction()
    with TaskManager():
        sl = HelmholtzSL(j_trial * ds(bonus_intorder=bonus_intorder), kappa, **bem_params)(j_h)
        dl = HelmholtzDL(m_trial * ds(bonus_intorder=bonus_intorder), kappa, **bem_params)(m_h)
        gradient_sl = grad(sl)
        return {
            'sl_out': complex(sl(point_out)), 'sl_in': complex(sl(point_in)),
            'dl_out': complex(dl(point_out)), 'dl_in': complex(dl(point_in)),
            'dn_sl_out': complex(np.dot(normal_h, gradient_sl(point_out))),
            'dn_sl_in': complex(np.dot(normal_h, gradient_sl(point_in))),
        }


def potential_integrands(point):
    tx, ty, tz = point
    dx, dy, dz = tx-x, ty-y, tz-z
    distance = sqrt(dx**2 + dy**2 + dz**2)
    source_normal_component = mesh_normal[0]*dx + mesh_normal[1]*dy + mesh_normal[2]*dz
    # The target normal is fixed at the surface anchor, on both approach sides.
    target_normal_component = normal_h[0]*dx + normal_h[1]*dy + normal_h[2]*dz
    wave = exp(1j*kappa*distance)
    sl_integrand = j_h * wave / (FOUR_PI * distance)
    radial_derivative = wave * (1j*kappa*distance - 1) / (FOUR_PI * distance**3)
    dl_integrand = -m_h * source_normal_component * radial_derivative
    dn_sl_integrand = j_h * target_normal_component * radial_derivative
    return {'sl': sl_integrand, 'dl': dl_integrand, 'dn_sl': dn_sl_integrand}


def evaluate_quadrature_pair(point_out, point_in, bonus_intorder, use_duffy):
    orders = {
        'sl': 2 + flux_order + bonus_intorder,
        'dl': 2 + trace_order + bonus_intorder,
        'dn_sl': 2 + flux_order + bonus_intorder,
    }
    values = {}
    with TaskManager():
        for side, point in (('out', point_out), ('in', point_in)):
            for component, integrand in potential_integrands(point).items():
                order = orders[component]
                if use_duffy:
                    value = duffy_quadrature.integrate(integrand, point, order)
                else: # gauss quadrature
                    value = Integrate(integrand, source_mesh, definedon=source_region, order=order)
                values[component + '_' + side] = complex(value)
    return values


def evaluate_gauss_pair(point_out, point_in, bonus_intorder):
    return evaluate_quadrature_pair(point_out, point_in, bonus_intorder, use_duffy=False)


def evaluate_duffy_pair(point_out, point_in, bonus_intorder):
    return evaluate_quadrature_pair(point_out, point_in, bonus_intorder, use_duffy=True)


def defects(values):
    double_jump = values['dl_out'] - values['dl_in']
    derivative_jump = values['dn_sl_out'] - values['dn_sl_in']
    single_jump = values['sl_out'] - values['sl_in']
    single_scale = max(abs(values['sl_out']), abs(values['sl_in']), 1e-15)
    return {
        'defect_dl_jump': abs(double_jump-m_at_surface) / abs(m_at_surface),
        'defect_dnsl_jump': abs(derivative_jump+j_at_surface) / abs(j_at_surface),
        'defect_sl_jump': abs(single_jump) / single_scale,
        'computed_dl_jump': double_jump,
        'exact_dl_jump': m_at_surface,
        'computed_dnsl_jump': derivative_jump,
        'exact_dnsl_jump': -j_at_surface,
        'computed_sl_jump': single_jump,
    }


def reference_errors(values, reference):
    """Compare the two SL values separately with the single fixed-order reference."""
    errors = {
        f'sl_{side}_reference_error': abs(values[f'sl_{side}'] - reference[f'sl_{side}'])
        / max(abs(reference[f'sl_{side}']), 1e-15)
        for side in ('out', 'in')
    }
    errors['sl_value_reference_error'] = max(errors.values())
    return errors


def evaluate_sl_reference(point_out, point_in):
    """Evaluate only SL, at the reference order configured at the notebook start."""
    j_trial = flux_space.TrialFunction()
    with TaskManager():
        sl = HelmholtzSL(j_trial * ds(bonus_intorder=reference_bonus_intorder), kappa, **bem_params)(j_h)
        return {'sl_out': complex(sl(point_out)), 'sl_in': complex(sl(point_in))}


## Jump experiment

The geometry, selected element, FE densities, and trace-space orders remain fixed. We vary the approach distance, quadrature bonus, and method. The main plots compare the double-layer jump and the SL normal-derivative jump directly with their exact limiting values. The SL continuity defect remains available as a secondary diagnostic.

The SL gradient is evaluated analytically through `grad(sl)` for singularity extraction and through the differentiated kernel for Gauss and Duffy quadrature. No finite differences of potential values are used. All methods use the same effective orders: `2 + flux_order + bonus` for SL and its normal derivative, and `2 + trace_order + bonus` for DL.

We retain the one-sided values and signed jumps. Only the separate SL value analysis uses a numerical reference: one singularity-extraction evaluation per target pair at the single `reference_bonus_intorder` fixed at the beginning of the notebook. Each SL value difference is normalized by the absolute reference value on the corresponding side, with a `1e-15` floor.


In [ ]:
rows = []
reference_rows = []
reference_values = {}
methods = (
    ('naive Gauss (G)', evaluate_gauss_pair),
    ('Duffy only (D)', evaluate_duffy_pair),
    ('singularity extraction (*)', evaluate_star_pair),
)
for delta in distances:
    point_out = surface_point + delta*normal_h
    point_in = surface_point - delta*normal_h
    reference = evaluate_sl_reference(point_out, point_in)
    reference_values[delta] = reference
    reference_rows.append({
        'delta': delta, 
        'bonus_intorder': reference_bonus_intorder,
        **reference,
    })
    for bonus in bonus_intorders:
        for method, evaluate in methods:
            values = evaluate(point_out, point_in, bonus)
            rows.append({
                'method': method, 
                'delta': delta, 
                'bonus_intorder': bonus,
                **values, **defects(values), 
                **reference_errors(values, reference),
            })

jump_results = pd.DataFrame(rows)
reference_results = pd.DataFrame(reference_rows)
jump_results.to_csv(OUTPUT_DIR / "helmholtz_jump_results.csv", index=False)
reference_results.to_csv(OUTPUT_DIR / "helmholtz_jump_reference.csv", index=False)
display(jump_results)


In [ ]:
styles = {
    'Duffy only (D)': ('D', ':', 'tab:green', '(D) Gauss + Duffy'),
    'naive Gauss (G)': ('s', '--', 'tab:orange', '(G) Gauss'),
    'singularity extraction (*)': ('o', '-', 'tab:blue', '(*) Analytical + Duffy + Gauss'),
}
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.7))
for method, (marker, linestyle, color, label) in styles.items():
    data = jump_results[(jump_results.method == method) & (jump_results.bonus_intorder == selected_bonus_intorder)].sort_values('delta')
    axes[0].loglog(data.delta, data.defect_dl_jump, marker=marker, linestyle=linestyle, color=color, label=label)
    axes[1].loglog(data.delta, data.defect_dnsl_jump, marker=marker, linestyle=linestyle, color=color, label=label)
for ax in axes:
    ax.set_xlabel('Approach distance')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best')
    ax.invert_xaxis()
axes[0].set_ylabel('relative defect')
axes[0].set_title('Double-layer jump defect')
axes[1].set_title('SL normal-derivative jump defect')
fig.suptitle(f'One-sided limits | bonus={selected_bonus_intorder}, geometry order={curve_order}, H1={trace_order}, SurfaceL2={flux_order}')
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "helmholtz_jump_defects_vs_distance.png", dpi=300)
plt.show()

closest_distance = min(distances)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.7))
for method, (marker, linestyle, color, label) in styles.items():
    data = jump_results[(jump_results.method == method) & (jump_results.delta == closest_distance)].sort_values('bonus_intorder')
    axes[0].semilogy(data.bonus_intorder, data.defect_dl_jump, marker=marker, linestyle=linestyle, color=color, label=label)
    axes[1].semilogy(data.bonus_intorder, data.defect_dnsl_jump, marker=marker, linestyle=linestyle, color=color, label=label)
for ax in axes:
    ax.set_xlabel('Quadrature bonus')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best')
axes[0].set_ylabel('relative defect')
axes[0].set_title('Double-layer jump defect')
axes[1].set_title('SL normal-derivative jump defect')
fig.suptitle(rf'Quadrature comparison at $\delta={closest_distance:.0e}$ | geometry order={curve_order}, H1={trace_order}, SurfaceL2={flux_order}')
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "helmholtz_jump_defects_vs_order.png", dpi=300)
plt.show()


In [ ]:
# The original SL values are compared separately, before taking any difference.
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.7))
for column, side in enumerate(('out', 'in')):
    error = 'sl_' + side + '_reference_error'
    for method, (marker, linestyle, color, label) in styles.items():
        order_data = jump_results[(jump_results.method == method) & (jump_results.delta == closest_distance)].sort_values('bonus_intorder')
        distance_data = jump_results[(jump_results.method == method) & (jump_results.bonus_intorder == selected_bonus_intorder)].sort_values('delta')
        axes[0, column].semilogy(order_data.bonus_intorder, order_data[error],
                                marker=marker, linestyle=linestyle, color=color, label=label)
        axes[1, column].loglog(distance_data.delta, distance_data[error],
                              marker=marker, linestyle=linestyle, color=color, label=label)
    side_label = 'exterior' if side == 'out' else 'interior'
    axes[0, column].set_title(f'SL {side_label}: distance = {closest_distance:.0e}')
    axes[1, column].set_title(f'SL {side_label}: bonus = {selected_bonus_intorder}')
    axes[0, column].set_xlabel('Quadrature bonus')
    axes[1, column].set_xlabel('Approach distance')
    axes[1, column].invert_xaxis()
for ax in axes.flat:
    ax.set_ylabel(r'$|S-S_{\mathrm{ref}}|/\max(|S_{\mathrm{ref}}|,10^{-15})$')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best', fontsize=9)
fig.suptitle(f'Helmholtz SL values: relative difference to the numerical reference\n'
             f'Reference: singularity extraction (*), bonus_intorder={reference_bonus_intorder} (same mesh and density)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'helmholtz_sl_value_errors.png', dpi=300)
fig.savefig(OUTPUT_DIR / 'helmholtz_sl_value_errors.pdf')
plt.show()


## Reading the results

**The jump plots do not use a numerical reference because the exact limiting jumps are known:** $[D]=m_h(y_0)$ and $[\partial_n S]=-j_h(y_0)$. The plotted defects compare directly with these values. An additional numerical reference is unnecessary for this test and would itself contain a finite-distance approach error.

At fixed $\delta>0$, even exact integration generally gives a nonzero defect relative to the limiting jump. A plateau as the quadrature order increases can therefore reflect the finite approach distance rather than insufficient integration accuracy. A plateau that decreases with $\delta$ supports this interpretation; roundoff and unresolved quadrature can also limit convergence.

For the single-layer potential, continuity alone is insufficient: nearly equal errors in the exterior and interior values can cancel in their difference. An accurate normal-derivative jump also does not establish the accuracy of the potential values themselves. We therefore compare the two SL values separately with **one numerical reference**, whose order is fixed by `reference_bonus_intorder` at the beginning of the notebook. This comparison uses the same FE geometry, density, and target points and avoids cancellation between sides.

The SL value-error plots show differences from this fixed-order numerical baseline, not certified errors against an exact potential value. They reveal how the methods and quadrature orders affect the individual values, while the jump plots independently test the exact boundary limits.
